## 1. Get the code and data

In [ ]:
!rm -rf court-timeliness-monitor!git clone https://github.com/nguyendoangiakhanh/court-timeliness-monitor.git%cd court-timeliness-monitor!pip install -q -r requirements.txt!python run_pipeline.py

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

processed_dir = Path("data/processed")
screenshot_dir = Path("outputs/colab_screenshot_charts")
screenshot_dir.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 130,
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

def save_and_show(fig, filename):
    path = screenshot_dir / filename
    fig.tight_layout()
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved to: {path}")

# 1. Monthly intake vs completion
monthly = pd.read_csv(processed_dir / "monthly_flow.csv")
monthly = monthly.dropna(subset=["month"]).sort_values("month")

fig, ax = plt.subplots()
ax.plot(monthly["month"], monthly["filings"], marker="o", linewidth=2, label="Filed")
ax.plot(monthly["month"], monthly["disposals"], marker="o", linewidth=2, label="Completed")
ax.set_title("Monthly volume: filed vs completed")
ax.set_xlabel("Month")
ax.set_ylabel("Number of records")
ax.legend(frameon=False)
plt.xticks(rotation=45, ha="right")
save_and_show(fig, "01_monthly_volume_simple.png")

In [ ]:
cases = pd.read_csv(processed_dir / "cases_clean.csv")

valid = cases[
    (cases["quality_flag"] == False) &
    (cases["waiting_days"].notna())
].copy()

median_days = valid["waiting_days"].median()
mean_days = valid["waiting_days"].mean()

fig, ax = plt.subplots()
ax.hist(valid["waiting_days"], bins=35, edgecolor="white")
ax.axvline(median_days, linestyle="-", linewidth=2, label=f"Median: {median_days:.0f} days")
ax.axvline(mean_days, linestyle="--", linewidth=2, label=f"Mean: {mean_days:.0f} days")
ax.set_title("Distribution of completion time")
ax.set_xlabel("Days")
ax.set_ylabel("Number of records")
ax.legend(frameon=False)
save_and_show(fig, "02_distribution_simple.png")

In [ ]:
region = pd.read_csv(processed_dir / "summary_by_region.csv")
region = region.dropna(subset=["median_days"]).sort_values("median_days")

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(region["region"], region["median_days"])
ax.bar_label(bars, fmt="%.0f", padding=4)
ax.set_title("Median completion time by region")
ax.set_xlabel("Median days")
ax.set_ylabel("")
save_and_show(fig, "03_region_simple.png")

In [ ]:
backlog = pd.read_csv(processed_dir / "backlog_age_profile.csv")

pivot = (
    backlog
    .pivot(index="region", columns="age_band", values="open_cases")
    .fillna(0)
)

order = ["0-3 months", "3-12 months", "12+ months"]
pivot = pivot[[col for col in order if col in pivot.columns]]

fig, ax = plt.subplots(figsize=(8.5, 4.5))
left = None

for col in pivot.columns:
    if left is None:
        ax.barh(pivot.index, pivot[col], label=col)
        left = pivot[col].copy()
    else:
        ax.barh(pivot.index, pivot[col], left=left, label=col)
        left += pivot[col]

ax.set_title("Open records by age band")
ax.set_xlabel("Number of open records")
ax.set_ylabel("")
ax.legend(frameon=False, loc="lower right")
save_and_show(fig, "04_open_age_band_simple.png")

In [ ]:
cleaning = pd.read_csv(processed_dir / "cleaning_log.csv")
cleaning_simple = cleaning.sort_values("rows_affected", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(cleaning_simple["issue"], cleaning_simple["rows_affected"])
ax.bar_label(bars, padding=4)
ax.set_title("Data-quality issues logged during cleaning")
ax.set_xlabel("Rows affected")
ax.set_ylabel("")
save_and_show(fig, "05_data_quality_simple.png")

## 6. The finding — recent "improvement" is incomplete dataMedian completion time can only be measured on records that have **closed**. Forrecently filed records, only the fast ones have closed, so the recent periods are abiased subset and the median falls. That reads as improvement when nothing has improved.The closure rate makes it explicit.

In [ ]:
quarter = pd.read_csv(processed_dir / "disposal_time_by_quarter.csv")e = quarter[quarter["region"] == "Eastern"].sort_values("filed_quarter")fig, ax = plt.subplots(figsize=(9.5, 5))ax.plot(e["filed_quarter"], e["median_days"], marker="o", lw=2,        color="grey", ls="--", label="Measured (all quarters)")reportable = e[e["complete"]]ax.plot(reportable["filed_quarter"], reportable["median_days"], marker="o", lw=2.5,        label="Reportable (>=80% closed)")incomplete = e[~e["complete"]]ax.axvspan(incomplete["filed_quarter"].iloc[0], e["filed_quarter"].iloc[-1],           alpha=0.12, color="orange")ax.text(incomplete["filed_quarter"].iloc[len(incomplete)//2], ax.get_ylim()[1]*0.96,        "under 80% of these\nfilings have closed",        ha="center", va="top", fontsize=9)ax.set_title("Recent 'improvement' is incomplete data, not faster processing")ax.set_xlabel("Filing quarter")ax.set_ylabel("Median days to completion")ax.legend(frameon=False)plt.xticks(rotation=45, ha="right")save_and_show(fig, "06_censoring_simple.png")print(e[["filed_quarter", "filed", "closed", "closure_rate", "complete"]].to_string(index=False))

Closure rate falls from about **94%** in the earliest quarter to about **21%** in themost recent. The pipeline flags anything under 80% as incomplete and reports the openbacklog age profile alongside instead — that measure *does* see the slow records,because it counts what is still waiting.

## 7. All charts together — for screenshotting

In [ ]:
from IPython.display import Image, display

for path in sorted(screenshot_dir.glob("*.png")):
    print(path.name)
    display(Image(filename=str(path), width=850))